In [74]:
import pandas as pd

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


In [75]:
df = pd.read_csv("/Users/andrewdrabkin/Documents/BLMA/all_data_cleaned.csv")



In [76]:
df.head()



,zcta,date,value_1_bed,value_2_bed,value_3_bed,value_4_bed,value_5_bed,all_homes,all_sfh,all_condo,...,new_listing_count,price_drops_pct_of_inventory,price_reduced_count,average_listing_price,median_listing_price_per_square_foot,active_listing_count_historical,total_listing_count_yy_smoothed,median_days_on_market_yy_smoothed,median_listing_price_yy_smoothed,average_listing_price_yy_smoothed
0,1001,2013-01-01,NaN,149000.0,202000.0,225000.0,250000.0,185000.0,199000.0,138000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1002,2013-01-01,126000.0,181000.0,271000.0,349000.0,426000.0,292000.0,314000.0,172000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1005,2013-01-01,NaN,136000.0,166000.0,197000.0,217000.0,164000.0,164000.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1007,2013-01-01,NaN,187000.0,249000.0,294000.0,313000.0,247000.0,251000.0,194000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1008,2013-01-01,NaN,167000.0,211000.0,244000.0,NaN,194000.0,196000.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [77]:
df.columns

Index(['zcta', 'date', 'value_1_bed', 'value_2_bed', 'value_3_bed',
       'value_4_bed', 'value_5_bed', 'all_homes', 'all_sfh', 'all_condo',
       'all_homes_projections', 'all_homes_prior', 'all_homes_yy_smoothed',
       'all_homes_200701', 'all_homes_201201', 'dollar_drop_2007_2012',
       'household_income', 'household_income_prior', 'household_income_delta',
       'pct_below_pov_line', 'pct_below_pov_line_prior',
       'pct_below_pov_line_delta', 'total_pop', 'total_pop_delta',
       'total_pop_prior', 'pct_wfh', 'pct_wfh_delta', 'pct_wfh_prior',
       'pct_0_bed_rentals', 'pct_1_bed_rentals', 'pct_2_bed_rentals',
       'pct_3_bed_rentals', 'pct_4_bed_rentals', 'pct_5_bed_rentals',
       'pct_0_bed_stock', 'pct_1_bed_stock', 'pct_2_bed_stock',
       'pct_3_bed_stock', 'pct_4_bed_stock', 'pct_5_bed_stock', 'pct_foreign',
       'pct_foreign_delta', 'pct_foreign_prior', 'pct_white',
       'pct_white_delta', 'pct_white_prior', 'pct_black', 'pct_black_delta',
       'pct_bl

Check Data range

In [78]:
start_date = df['date'].min()
end_date = df['date'].max()

print(start_date, end_date)

2013-01-01 2025-08-01


In [79]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])


Check to see that only zip/date pair happened once

In [80]:
dupes = df.duplicated(subset=['zcta', 'date'])

dupes.any()


np.False_

In [81]:
df = df.sort_values(["zcta", "date"])


In [82]:
features = []  # no covariates
target = "median_listing_price"

# MODEL NUMBER 1 - one single average price for everyone

In [83]:
df["date"].dtype


dtype('<M8[ns]')

In [84]:
df_model = df.dropna(subset=["zcta", "date", target]).copy()
df_model["year"] = df_model["date"].dt.year

results = []

for test_year in range(2014, 2024):
    train = df_model[df_model["year"] <= test_year - 1]
    test  = df_model[df_model["year"] == test_year]

    if len(train) == 0 or len(test) == 0:
        continue

    y_train = train[target]
    y_test  = test[target]

    # Intercept-only prediction
    y_train_pred = np.repeat(y_train.mean(), len(y_train))
    y_test_pred  = np.repeat(y_train.mean(), len(y_test))

    results.append({
        "test_year": test_year,
        "train_rmse": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "test_rmse":  np.sqrt(mean_squared_error(y_test,  y_test_pred)),
        "n_train": len(train),
        "n_test": len(test),
    })

results_df = pd.DataFrame(results)
results_df


,test_year,train_rmse,test_rmse,n_train,n_test
0,2017,321481.629314,349214.870703,152877,305389
1,2018,340095.686864,364997.215101,458266,305613
2,2019,350199.106212,399578.234474,763879,303387
3,2020,364858.694649,477779.034291,1067266,301942
4,2021,392443.230676,704342.771510,1369208,296668
5,2022,463276.322485,739331.590643,1665876,297482
6,2023,514340.608471,781166.711256,1963358,298111


# Model 2 - Different average price for each ZIP code

In [85]:


# Assume df_model already exists with:
# columns: zcta, date, year, median_listing_price

target = "median_listing_price"

results = []

for test_year in range(start_year + 1, end_year + 1):

    train = df_model[df_model["year"] <= test_year - 1]
    test  = df_model[df_model["year"] == test_year]

    if len(train) == 0 or len(test) == 0:
        continue

    # 1) Compute ZIP-level mean prices from TRAIN ONLY
    zip_means = (
        train
        .groupby("zcta")[target]
        .mean()
    )

    # 2) Predict: lookup ZIP mean
    y_train_pred = train["zcta"].map(zip_means)
    y_test_pred  = test["zcta"].map(zip_means)

    # 3) Handle ZIPs never seen in training (rare but possible)
    global_mean = train[target].mean()

    y_train_pred = y_train_pred.fillna(global_mean)
    y_test_pred  = y_test_pred.fillna(global_mean)

    # 4) Compute errors
    train_mse = mean_squared_error(train[target], y_train_pred)
    test_mse  = mean_squared_error(test[target],  y_test_pred)

    results.append({
        "test_year": test_year,
        "n_train": len(train),
        "n_test": len(test),
        "train_rmse": np.sqrt(train_mse),
        "test_rmse":  np.sqrt(test_mse),
    })

results_zip_mean = pd.DataFrame(results).sort_values("test_year").reset_index(drop=True)

print(results_zip_mean)
print("\nAverage test RMSE:", results_zip_mean["test_rmse"].mean())


   test_year  n_train  n_test     train_rmse      test_rmse
0       2017   152877  305389   76860.142655  148114.249672
1       2018   458266  305613  105635.483762  179247.572588
2       2019   763879  303387  132942.669689  241501.644930
3       2020  1067266  301942  161737.463287  322067.634340
4       2021  1369208  296668  203131.312625  570532.015240
5       2022  1665876  297482  292952.310047  545896.792133
6       2023  1963358  298111  338158.135224  576178.816454
7       2024  2261469  301387  374327.264480  407763.939398
8       2025  2562856  199786  376726.194553  361441.707041

Average test RMSE: 372527.1524217687


### DOING IT BY ZIP

In [86]:
oc_zips = [
    "90620", "90621", "90623", "90630",
    "90631", "90632",
    "90720",

    "92602", "92603", "92604", "92606", "92610",
    "92612", "92614", "92617", "92618", "92620",
    "92624", "92625", "92626", "92627", "92629",
    "92630", "92646", "92647", "92648", "92649",
    "92651", "92653", "92655", "92656", "92657",
    "92660", "92661", "92662", "92663",
    "92672", "92673", "92675", "92676", "92677",
    "92678", "92679",

    "92701", "92703", "92704", "92705", "92706",
    "92707", "92708", "92780", "92782",

    "92801", "92802", "92803", "92804", "92805",
    "92806", "92807", "92808",
    "92821", "92823",
    "92831", "92832", "92833", "92835",
    "92840", "92841", "92843", "92844", "92845",
    "92861", "92865", "92867", "92868", "92869"
]

oc_df = df[df["zcta"].astype(str).isin(oc_zips)].copy()


In [87]:
oc_df["date"] = pd.to_datetime(oc_df["date"], errors="coerce")
oc_df = oc_df.dropna(subset=["date", "zcta", "median_listing_price"]).copy()
oc_df = oc_df.sort_values(["date", "zcta"]).copy()

oc_df["year"] = oc_df["date"].dt.year

In [88]:
print(oc_df["date"].min(), oc_df["date"].max())
print(len(oc_df), "rows")


2016-07-01 00:00:00 2025-08-01 00:00:00
7920 rows


In [89]:
target = "median_listing_price"

start_year = oc_df["year"].min()
end_year   = oc_df["year"].max()


MODEL 0 — OC ZIP-mean baseline (re-run on OC)

In [90]:
results_zip_mean = []

for test_year in range(start_year + 1, end_year + 1):

    train = oc_df[oc_df["year"] <= test_year - 1]
    test  = oc_df[oc_df["year"] == test_year]

    if len(train) == 0 or len(test) == 0:
        continue

    zip_means = train.groupby("zcta")[target].mean()
    global_mean = train[target].mean()

    y_train_pred = train["zcta"].map(zip_means).fillna(global_mean)
    y_test_pred  = test["zcta"].map(zip_means).fillna(global_mean)

    results_zip_mean.append({
        "test_year": test_year,
        "train_rmse": np.sqrt(mean_squared_error(train[target], y_train_pred)),
        "test_rmse":  np.sqrt(mean_squared_error(test[target],  y_test_pred)),
        "n_train": len(train),
        "n_test": len(test),
    })

oc_zip_mean = pd.DataFrame(results_zip_mean)
oc_zip_mean


,test_year,train_rmse,test_rmse,n_train,n_test
0,2017,79393.291613,1.405281e+05,432,864
1,2018,100773.499831,1.502519e+05,1296,864
2,2019,113499.432478,1.626057e+05,2160,864
3,2020,124122.243793,2.456977e+05,3024,864
4,2021,153762.318048,5.599235e+05,3888,864
5,2022,260147.996565,8.058919e+05,4752,864
6,2023,380941.362063,1.099333e+06,5616,864
7,2024,519320.915001,1.131185e+06,6480,864
8,2025,610891.055324,7.328606e+05,7344,576


MODEL 1 — Log-price ZIP-mean (recommended baseline)

In [91]:
oc_lp = oc_df[oc_df[target] > 0].copy()
oc_lp["log_price"] = np.log(oc_lp[target])

results_zip_mean_log = []

for test_year in range(start_year + 1, end_year + 1):

    train = oc_lp[oc_lp["year"] <= test_year - 1]
    test  = oc_lp[oc_lp["year"] == test_year]

    if len(train) == 0 or len(test) == 0:
        continue

    zip_means_log = train.groupby("zcta")["log_price"].mean()
    global_mean_log = train["log_price"].mean()

    y_train_pred = train["zcta"].map(zip_means_log).fillna(global_mean_log)
    y_test_pred  = test["zcta"].map(zip_means_log).fillna(global_mean_log)

    results_zip_mean_log.append({
        "test_year": test_year,
        "train_rmse_log": np.sqrt(mean_squared_error(train["log_price"], y_train_pred)),
        "test_rmse_log":  np.sqrt(mean_squared_error(test["log_price"],  y_test_pred)),
        "n_test": len(test),
    })

oc_zip_mean_log = pd.DataFrame(results_zip_mean_log)
oc_zip_mean_log


,test_year,train_rmse_log,test_rmse_log,n_test
0,2017,0.055304,0.109482,864
1,2018,0.075594,0.109919,864
2,2019,0.084703,0.102246,864
3,2020,0.086961,0.141514,864
4,2021,0.098201,0.232869,864
5,2022,0.127741,0.317873,864
6,2023,0.165116,0.359010,864
7,2024,0.196941,0.370447,864
8,2025,0.220436,0.349755,576


MODEL 2 — Simple OC linear regression (no ZIP effects)

In [92]:
features = [
    "household_income",
    "total_pop",
    "pct_with_bachelor",
    "pct_vacancy_prior",
]


In [93]:
results_linear = []

for test_year in range(start_year + 1, end_year + 1):

    train = oc_df[oc_df["year"] <= test_year - 1].copy()
    test  = oc_df[oc_df["year"] == test_year].copy()

    train = train.dropna(subset=features + [target])
    test  = test.dropna(subset=features + [target])

    if len(train) == 0 or len(test) == 0:
        continue

    X_train, y_train = train[features], train[target]
    X_test,  y_test  = test[features],  test[target]

    model = LinearRegression()
    model.fit(X_train, y_train)

    results_linear.append({
        "test_year": test_year,
        "train_rmse": np.sqrt(mean_squared_error(y_train, model.predict(X_train))),
        "test_rmse":  np.sqrt(mean_squared_error(y_test,  model.predict(X_test))),
        "n_test": len(test),
    })

oc_linear = pd.DataFrame(results_linear)
oc_linear


,test_year,train_rmse,test_rmse,n_test
0,2019,378171.423839,3.843474e+05,72
1,2020,373443.861661,5.044283e+05,72
2,2021,420062.917936,5.976966e+05,72
3,2022,469682.383600,1.005376e+06,72
4,2023,608035.797334,8.749126e+05,72


MODEL 3 — ZIP-demeaned log-price linear regression

In [94]:
target = "median_listing_price"

oc_lp = oc_df.dropna(subset=[target]).copy()
oc_lp = oc_lp[oc_lp[target] > 0].copy()
oc_lp["log_price"] = np.log(oc_lp[target])

Test to see rmse decreases with features

In [95]:
features = [
    "household_income",
    "total_pop",
    "pct_with_bachelor",
]


In [96]:
results_zip_dm_log = []

for test_year in range(start_year + 1, end_year + 1):

    train = oc_lp[oc_lp["year"] <= test_year - 1].copy()
    test  = oc_lp[oc_lp["year"] == test_year].copy()

    # Drop missing feature rows *inside* each split
    train = train.dropna(subset=features + ["log_price"])
    test  = test.dropna(subset=features + ["log_price"])

    if len(train) == 0 or len(test) == 0:
        continue

    # 1) ZIP mean log prices (TRAIN ONLY)
    zip_means_log = train.groupby("zcta")["log_price"].mean()
    global_mean_log = train["log_price"].mean()

    # 2) De-mean log price
    y_train_dm = train["log_price"] - train["zcta"].map(zip_means_log)
    y_test_dm  = test["log_price"]  - test["zcta"].map(zip_means_log)

    # Fallback for unseen ZIPs
    y_test_dm = y_test_dm.fillna(test["log_price"] - global_mean_log)

    # 3) Fit linear model on deviations
    X_train = train[features]
    X_test  = test[features]

    model = LinearRegression()
    model.fit(X_train, y_train_dm)

    # 4) Predict deviations
    train_dev_pred = model.predict(X_train)
    test_dev_pred  = model.predict(X_test)

    # 5) Reconstruct log-price predictions
    y_train_pred_log = train["zcta"].map(zip_means_log) + train_dev_pred
    y_test_pred_log  = test["zcta"].map(zip_means_log)  + test_dev_pred
    y_test_pred_log  = y_test_pred_log.fillna(global_mean_log)

    # 6) Evaluate in log space
    train_rmse_log = np.sqrt(mean_squared_error(train["log_price"], y_train_pred_log))
    test_rmse_log  = np.sqrt(mean_squared_error(test["log_price"],  y_test_pred_log))

    results_zip_dm_log.append({
        "test_year": test_year,
        "train_rmse_log": train_rmse_log,
        "test_rmse_log": test_rmse_log,
        "test_pct_error_approx": np.expm1(test_rmse_log),
        "n_train": len(train),
        "n_test": len(test),
    })

oc_zip_dm_log = pd.DataFrame(results_zip_dm_log)
oc_zip_dm_log


,test_year,train_rmse_log,test_rmse_log,test_pct_error_approx,n_train,n_test
0,2018,0.000000,0.142484,0.153134,72,72
1,2019,0.071220,0.104490,0.110144,144,72
2,2020,0.076103,0.173326,0.189254,216,72
3,2021,0.099435,0.171028,0.186524,288,72
4,2022,0.112056,0.299139,0.348697,360,72
5,2023,0.149162,0.285779,0.330799,432,72


Model 4

In [97]:
target = "median_listing_price"

oc_lp = oc_df.dropna(subset=[target]).copy()
oc_lp = oc_lp[oc_lp[target] > 0].copy()
oc_lp["log_price"] = np.log(oc_lp[target])

# Monthly time index (no leakage)
min_date = oc_lp["date"].min()
oc_lp["time_index"] = (
    (oc_lp["date"].dt.year - min_date.year) * 12
    + (oc_lp["date"].dt.month - min_date.month)
)


features = [
    "time_index",
    "median_days_on_market",
    "active_listing_count",
    "new_listing_count",
    "price_drops_pct_of_inventory",
]


In [98]:
results_model4 = []

for test_year in range(start_year + 1, end_year + 1):

    train = oc_lp[oc_lp["year"] <= test_year - 1].copy()
    test  = oc_lp[oc_lp["year"] == test_year].copy()

    # Drop missing rows *inside* each split
    train = train.dropna(subset=features + ["log_price"])
    test  = test.dropna(subset=features + ["log_price"])

    if len(train) == 0 or len(test) == 0:
        continue

    # ZIP mean log prices (TRAIN ONLY)
    zip_means_log = train.groupby("zcta")["log_price"].mean()
    global_mean_log = train["log_price"].mean()

    # ZIP-demeaned targets
    y_train_dm = train["log_price"] - train["zcta"].map(zip_means_log)
    y_test_dm  = test["log_price"]  - test["zcta"].map(zip_means_log)

    y_test_dm = y_test_dm.fillna(test["log_price"] - global_mean_log)

    # Design matrices
    X_train = train[features]
    X_test  = test[features]

    # Fit linear model
    model = LinearRegression()
    model.fit(X_train, y_train_dm)

    # Predict deviations
    train_dev_pred = model.predict(X_train)
    test_dev_pred  = model.predict(X_test)

    # Reconstruct log-price predictions
    y_train_pred_log = train["zcta"].map(zip_means_log) + train_dev_pred
    y_test_pred_log  = test["zcta"].map(zip_means_log)  + test_dev_pred
    y_test_pred_log  = y_test_pred_log.fillna(global_mean_log)

    # Evaluate in log space
    train_rmse_log = np.sqrt(mean_squared_error(train["log_price"], y_train_pred_log))
    test_rmse_log  = np.sqrt(mean_squared_error(test["log_price"],  y_test_pred_log))

    results_model4.append({
        "test_year": test_year,
        "train_rmse_log": train_rmse_log,
        "test_rmse_log": test_rmse_log,
        "test_pct_error_approx": np.expm1(test_rmse_log),
        "n_train": len(train),
        "n_test": len(test),
    })

oc_model4 = pd.DataFrame(results_model4)
oc_model4


,test_year,train_rmse_log,test_rmse_log,test_pct_error_approx,n_train,n_test
0,2017,0.054203,0.092808,0.097251,432,864
1,2018,0.070222,0.102530,0.107971,1296,864
2,2019,0.078061,0.105297,0.111040,2160,864
3,2020,0.081358,0.117632,0.124831,3024,864
4,2021,0.088395,0.165874,0.180424,3888,864
5,2022,0.100983,0.200884,0.222483,4752,864
6,2023,0.114882,0.199123,0.220332,5616,862
7,2024,0.126148,0.161128,0.174836,6478,864
8,2025,0.129359,0.155951,0.168769,7342,576
